# T0 · Fit the baseline
### A fitted MMM is a *hypothesis*, not an answer

*Notebook 1 of 7 — The Experimental Measurement Lifecycle.*

We are at the top of the loop. In the [overview](lifecycle_00_overview.ipynb) we
argued that history alone can't tell you what your channels *cause* — because
marketers spend into demand, so spend and sales rise together for reasons that
have nothing to do with advertising working. This notebook builds the model that
every later stage refines: **Northwind Outfitters'** baseline marketing-mix model,
fit to three years of weekly national data across **TV, Search, Social, and
Display**.

Think of this fit as the **scaffold**. It is genuinely useful — it gives us a
coherent, uncertainty-aware read on every channel at once — but it is built
entirely from *observation*, so it can only ever report what **correlated** with
sales. By the end of this notebook you'll see two limits at once: the model is
**uncertain** (wide intervals, which it will happily tell you about), *and* it is
**biased** (systematically off, which it will not). The loop exists to close the
second gap, one experiment at a time.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os
os.environ.setdefault("TQDM_DISABLE", "1")   # quiet sampling progress bars (keeps outputs clean)
import sys
# Make the shared module importable whether the kernel runs from nbs/ or repo root.
for _p in (".", os.path.join(os.getcwd(), "nbs"), os.path.dirname(os.getcwd())):
    if os.path.isfile(os.path.join(_p, "lifecycle_common.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 140)

import logging
from loguru import logger
logger.disable("mmm_framework")
for _n in ("pymc", "pymc.sampling", "numpyro", "jax", "arviz"):
    logging.getLogger(_n).setLevel(logging.ERROR)

import lifecycle_common as L
print(f"{L.BRAND} — {L.TAGLINE}")

Northwind Outfitters — a national outdoor-apparel brand


## The world, and why we can grade it

Northwind's data is the framework's **`clean` synthetic world** — a brand whose
sales really were generated by the *exact* model family we're about to fit:

- **geometric adstock** per channel (this week's spend keeps working for a few
  weeks — advertising has a memory),
- **logistic saturation** per channel (each extra dollar buys a little less than
  the last — channels get crowded),
- a **Price** control, at **weekly** cadence.

Because the world and the model agree on structure, recovery is *honest*: any
error we see is estimation and confounding, not a rigged mismatch. And the world
ships with a **sealed answer key** — the true per-channel return — that the
analyst never sees while modelling. We only unseal it at the very end, as
narrator, to *grade* the fit. Everywhere else in this series, the truth stays in
the envelope.

In [2]:
# T0: load Northwind's baseline national MMM. The fit is done once and cached by
# lifecycle_common (a fast, real 2-chain x 300-draw NUTS posterior) — later
# notebooks reload this same posterior instead of re-sampling.
base = L.fit_baseline()
model, truth, mff_df = base["model"], base["truth"], base["mff_df"]

first, last = L.dataset_period(mff_df)
n_weeks = mff_df["Period"].nunique()
print(f"channels : {', '.join(truth['channels'])}   |   KPI: {L.KPI}   |   control: {L.CONTROL}")
print(f"window   : {first} -> {last}   ({n_weeks} weekly observations)")

# Convergence sanity — did the sampler actually agree with itself? R-hat near
# 1.0 means the independent chains explored the same posterior; ESS is the
# effective number of independent draws behind each number.
import arviz as az
diag = az.summary(model._trace, var_names=[f"beta_{c}" for c in L.CHANNELS], kind="diagnostics")
diag.index = [ix.replace("beta_", "") for ix in diag.index]
print("\nper-channel effect (beta) convergence:")
print(diag[["r_hat", "ess_bulk"]].round(3).to_string())
print(f"\ndraws: {model._trace.posterior.sizes['draw']}  chains: {model._trace.posterior.sizes['chain']}")

assert model._trace.posterior.sizes["draw"] >= 100        # a real posterior, not a point estimate
assert float(diag["r_hat"].max()) < 1.3                   # chains converged (loose bound for a fast teaching fit)

[Northwind Outfitters] loaded cached baseline fit from /var/folders/nw/92nd031j7p5d2bs8grpysp4w0000gn/T/mmm_lifecycle_cache
channels : TV, Search, Social, Display   |   KPI: Sales   |   control: Price
window   : 2021-01-04 -> 2023-12-25   (156 weekly observations)

per-channel effect (beta) convergence:
         r_hat  ess_bulk
TV        1.02     211.0
Search    1.02     309.0
Social    1.00     329.0
Display   1.00     238.0

draws: 300  chains: 2


R-hat sits right around **1.0** for every channel — the two chains agree, so the
posterior is trustworthy to *read* (this is a deliberately small, fast teaching
fit; a production run would use more draws for tighter effective sample sizes).

Converged, though, only means *"the model is sure about what the model believes."*
It says nothing about whether what the model believes is **right**. Let's look at
what it believes — starting with the number every analyst reaches for first.

In [3]:
# The analyst's headline view: average ROI per channel, WITH its uncertainty.
# roi_mean is the point estimate; [roi_hdi_low, roi_hdi_high] is the 94% credible
# interval — the band the model itself considers plausible.
from mmm_framework.reporting.helpers.roi import compute_roi_with_uncertainty

roi = compute_roi_with_uncertainty(model).set_index("channel").loc[L.CHANNELS]

fig = go.Figure()
for ch in L.CHANNELS:
    r = roi.loc[ch]
    fig.add_trace(go.Scatter(                              # the credible-interval band
        x=[r["roi_hdi_low"], r["roi_hdi_high"]], y=[ch, ch], mode="lines",
        line=dict(color=L.PALETTE[ch], width=9), opacity=0.4, showlegend=False))
    fig.add_trace(go.Scatter(                              # the point estimate
        x=[r["roi_mean"]], y=[ch], mode="markers", showlegend=False,
        marker=dict(color=L.PALETTE[ch], size=14, line=dict(color="white", width=1.5))))
fig.add_vline(x=1.0, line=dict(color=L.INK, width=1, dash="dot"),
              annotation_text="break-even ($1 back per $1)", annotation_position="top")
L.style(fig, title="Baseline MMM — average ROI per channel, with 94% credible interval")
fig.update_layout(xaxis_title="return per $1 of spend (average ROI)", yaxis_title="")
fig.show()

print(roi[["roi_mean", "roi_hdi_low", "roi_hdi_high", "prob_positive"]].round(3).to_string())
widths = (roi["roi_hdi_high"] - roi["roi_hdi_low"])
print(f"\nnarrowest 94% interval spans {widths.min():.2f} of ROI — none of these is a point answer.")
assert widths.min() > 0.1                                 # every channel carries real, non-trivial uncertainty

         roi_mean  roi_hdi_low  roi_hdi_high  prob_positive
channel                                                    
TV          0.497        0.353         0.614            1.0
Search      0.824        0.644         1.037            1.0
Social      0.684        0.480         0.898            1.0
Display     0.643        0.422         0.872            1.0

narrowest 94% interval spans 0.26 of ROI — none of these is a point answer.


Two things jump out. First, the bands are **wide** — the narrowest one still spans
a large chunk of ROI, so "the answer" is really a *range*. Second, the averages
sit **below break-even** ($1 back per $1). That is *normal* for a saturated media
model and not a red flag: **average** ROI is dragged down by the crowded, heavily
spent weeks, while the decision that matters — where the *next* dollar goes — runs
on **marginal** return. We'll get to marginal ROI when we allocate (T4). For now,
the point is only that a single fit answers with intervals, not verdicts.

In [4]:
# Where does the model think Northwind's sales come from? Each channel's mean
# contribution is its estimated share of modeled sales over the whole window.
contrib = roi["contribution_mean"]
share = 100 * contrib / contrib.sum()

fig = go.Figure(go.Bar(
    x=L.CHANNELS, y=[float(contrib[c]) for c in L.CHANNELS],
    marker_color=[L.PALETTE[c] for c in L.CHANNELS],
    text=[f"{share[c]:.0f}% of media" for c in L.CHANNELS], textposition="outside",
    cliponaxis=False, showlegend=False))
L.style(fig, title="Where the model thinks the sales come from (mean channel contribution)")
fig.update_layout(xaxis_title="", yaxis_title=f"modeled {L.KPI} contribution ($000s)")
fig.show()

print("mean contribution ($000s) and share of total media effect:")
for c in L.CHANNELS:
    print(f"  {c:<7} {L.dollars(contrib[c])}   ({share[c]:.0f}%)")
assert (contrib > 0).all()                                # the model credits every channel with positive lift

mean contribution ($000s) and share of total media effect:
  TV      $4,851   (32%)
  Search  $4,725   (31%)
  Social  $3,162   (21%)
  Display $2,487   (16%)


So the model has a full story: every channel pulls its weight, TV and Search
carry the largest slabs of modeled sales, and each ROI comes with an honest
error bar. If this were the end of the road, an analyst would allocate off this
picture. **Here is why they shouldn't** — and it's the crux of the whole series.

## The honesty check (and the case for T1)

Now we break the fourth wall. As *narrator* — never as the analyst — let's unseal
the answer key and lay the model's ROI next to the **truth** it was trying to
recover. Watch the direction of the gaps.

In [5]:
est = np.array([float(roi.loc[c, "roi_mean"]) for c in L.CHANNELS])
tru = np.array([float(truth["true_roas"][c]) for c in L.CHANNELS])

fig = go.Figure()
for ch in L.CHANNELS:                                     # faded band = what the model thought plausible
    r = roi.loc[ch]
    fig.add_trace(go.Scatter(
        x=[r["roi_hdi_low"], r["roi_hdi_high"]], y=[ch, ch], mode="lines",
        line=dict(color=L.PALETTE[ch], width=9), opacity=0.30, showlegend=False))
fig.add_trace(go.Scatter(                                 # the model's estimate
    x=est, y=L.CHANNELS, mode="markers", name="model estimate",
    marker=dict(color=[L.PALETTE[c] for c in L.CHANNELS], size=14, line=dict(color="white", width=1.5))))
fig.add_trace(go.Scatter(                                 # the sealed truth
    x=tru, y=L.CHANNELS, mode="markers", name="sealed truth",
    marker=dict(color=L.GOOD, symbol="diamond", size=15, line=dict(color="white", width=1.5))))
L.style(fig, title="Estimate vs. sealed truth — the gaps all point the same way",
        legend=dict(orientation="h", yanchor="top", y=-0.16, x=0))
fig.update_layout(xaxis_title="return per $1 of spend (average ROI)", yaxis_title="")
fig.show()

under = int((est < tru).sum())
miss = int(sum(not (roi.loc[c, "roi_hdi_low"] <= truth["true_roas"][c] <= roi.loc[c, "roi_hdi_high"])
               for c in L.CHANNELS))
print(f"under-credited channels : {under} of {len(L.CHANNELS)}   (model estimate below the truth)")
print(f"94% interval MISSES truth: {miss} of {len(L.CHANNELS)}   (truth outside the band the model trusted)")
print(f"mean ROI  — model {est.mean():.3f}  vs  truth {tru.mean():.3f}")
assert est.mean() < tru.mean()                            # the observational fit SYSTEMATICALLY under-credits

under-credited channels : 4 of 4   (model estimate below the truth)
94% interval MISSES truth: 1 of 4   (truth outside the band the model trusted)
mean ROI  — model 0.662  vs  truth 0.764


Every gap points the **same way**: the observational fit *under-credits* the
channels, and for at least one channel the truth falls **outside** the 94%
interval the model swore by. This is the failure the overview warned about, made
concrete — with a demand signal quietly moving spend and sales together, the
baseline hands some of advertising's real credit to "the baseline," and no amount
of extra draws or tighter chains will fix it. **The bias is invisible from
inside the model**: the credible interval only knows about the noise the model
can see, not the confounding baked into the history.

**Readout →** *We have a coherent, uncertainty-aware hypothesis — and it is wide,
and biased in a direction we could not have detected from the fit alone. We can't
close every gap: experiments cost real money and time. So the question isn't
"which channel are we most **unsure** about?" — it's "which channel, if we
measured it, would most **change the budget**?" Answering that is where the loop
earns its keep.*

That distinction — **information** vs. the **value** of information — is
[**T1 · Prioritize** → `lifecycle_02_prioritize`](lifecycle_02_prioritize.ipynb),
next. To come back to the map, see
[`lifecycle_00_overview`](lifecycle_00_overview.ipynb).